In [ ]:
from pathlib import Path
import argparse
import json
import re
import unicodedata
import pandas as pd

try:
    from unidecode import unidecode as _transliterate
except ImportError:
    def _transliterate(text: str) -> str:
        return text

In [ ]:
INPUT_FILE = Path("dataset/train/train_source1.tsv")
OUTPUT_DIR = Path("processed/source1_cleaned")
CHUNK_SIZE = 100_000
REQUIRED_COLUMNS = ["entity_id", "business_name", "business_address", "country"]
BASE_PIPELINE_VERSION = "cleaning_v2_advanced"
PIPELINE_VERSION = f"source1_{BASE_PIPELINE_VERSION}"

In [ ]:
from pathlib import Path
import argparse
import json
import re
import unicodedata
import pandas as pd

try:
    from unidecode import unidecode as _transliterate
except ImportError:
    def _transliterate(text: str) -> str:
        return text

In [ ]:
# ============================================================
# NORMALIZATION RULES
# ============================================================
NAME_ABBREVIATIONS = {"pvt": "private", "ltd": "limited", "inc": "incorporated", "corp": "corporation", "co": "company", "st": "saint"}
ADDRESS_ABBREVIATIONS = {"rd": "road", "st": "street", "ave": "avenue", "blvd": "boulevard", "dr": "drive", "ln": "lane", "hwy": "highway", "apt": "apartment", "opp": "opposite", "nr": "near"}
CORE_REMOVE_WORDS = {"company", "co", "corporation", "corp", "inc", "incorporated", "limited", "ltd", "private", "pvt", "llc", "llp", "plc", "sarl", "sas", "sasu", "eurl", "sa"}
LEADING_STOPWORDS = {"the"}
LANDMARK_PATTERN = re.compile(r"\b(near|opposite|behind|beside|next to|close to|adjacent to|in front of)\b")
MULTI_SPACE_RE = re.compile(r"\s+")
NUMBER_RE = re.compile(r"\d+[a-z]?(?:[/-]\d+[a-z]?)*")
PURE_DIGIT_TOKEN_RE = re.compile(r"\b\d{4,6}\b")
WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
STATS = {"total_rows": 0, "empty_name": 0, "empty_address": 0, "postal_code_found": 0, "landmark_found": 0}

In [ ]:
def unicode_normalize(text: str) -> str:
    if not text:
        return ""
    return unicodedata.normalize("NFKC", text)


def normalize_common(text: str) -> str:
    text = unicode_normalize(text)
    if not text:
        return ""
    text = text.lower().replace("&", " and ").replace("'", " ").replace('"', " ")
    text = re.sub(r"[^\w\s/-]", " ", text, flags=re.UNICODE)
    return MULTI_SPACE_RE.sub(" ", text).strip()


def expand_abbreviations(text: str, abbreviation_map: dict) -> str:
    if not text:
        return ""
    return " ".join(abbreviation_map.get(token, token) for token in text.split())


def clean_name(text: str) -> str:
    return expand_abbreviations(normalize_common(text), NAME_ABBREVIATIONS)


def clean_address(text: str) -> str:
    return expand_abbreviations(normalize_common(text), ADDRESS_ABBREVIATIONS)


def make_name_core(cleaned_name: str) -> str:
    if not cleaned_name:
        return ""
    tokens = cleaned_name.split()
    if tokens and tokens[0] in LEADING_STOPWORDS:
        tokens = tokens[1:]
    while tokens and tokens[-1] in CORE_REMOVE_WORDS:
        tokens.pop()
    return " ".join(tokens)


def make_name_latin(name_full_clean: str) -> str:
    if not name_full_clean or name_full_clean.isascii():
        return name_full_clean
    return _transliterate(name_full_clean)


def split_landmark(address_clean: str):
    if not address_clean:
        return "", ""
    match = LANDMARK_PATTERN.search(address_clean)
    if not match:
        return address_clean, ""
    return address_clean[:match.start()].strip(), address_clean[match.start():].strip()


def extract_postal_code(address_clean: str, country: str) -> str:
    digit_tokens = PURE_DIGIT_TOKEN_RE.findall(address_clean)
    if not digit_tokens:
        return ""
    country_upper = (country or "").strip().upper()
    if country_upper in {"INDIA", "IN"}:
        candidates = [token for token in digit_tokens if len(token) == 6]
    elif country_upper in {"US", "USA", "UNITED STATES", "UNITED STATES OF AMERICA"}:
        candidates = [token for token in digit_tokens if len(token) == 5]
    else:
        candidates = []
    return (candidates or digit_tokens)[-1]


def extract_numbers(text: str) -> str:
    if not text:
        return ""
    seen = set()
    values = []
    for value in NUMBER_RE.findall(text):
        if value not in seen:
            seen.add(value)
            values.append(value)
    return "|".join(values)


def tokenize(text: str) -> str:
    return "|".join(WORD_RE.findall(text)) if text else ""
def get_config():
    return {"pipeline_version": PIPELINE_VERSION, "input_file": str(INPUT_FILE), "input_size": INPUT_FILE.stat().st_size, "input_mtime": INPUT_FILE.stat().st_mtime, "chunk_size": CHUNK_SIZE, "required_columns": REQUIRED_COLUMNS}


def validate_or_create_manifest():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = OUTPUT_DIR / "manifest.json"
    current_config = get_config()
    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as f:
            old_config = json.load(f)
        for key in ["pipeline_version", "input_file", "input_size", "input_mtime", "chunk_size"]:
            if old_config.get(key) != current_config.get(key):
                raise RuntimeError(f"Manifest mismatch for '{key}'.")
    else:
        temp_manifest = OUTPUT_DIR / "manifest.json.tmp"
        with open(temp_manifest, "w", encoding="utf-8") as f:
            json.dump(current_config, f, indent=2)
        temp_manifest.replace(manifest_path)
    return manifest_path
def process_chunk(chunk: pd.DataFrame, chunk_id: int):
    final_file = OUTPUT_DIR / f"part_{chunk_id:05d}.parquet"
    done_file = OUTPUT_DIR / f"part_{chunk_id:05d}.done"
    if final_file.exists() and done_file.exists():
        print(f"[SKIP] Chunk {chunk_id}")
        return
    missing = [column for column in REQUIRED_COLUMNS if column not in chunk.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    chunk["business_name_original"] = chunk["business_name"].fillna("").astype(str)
    chunk["business_address_original"] = chunk["business_address"].fillna("").astype(str)
    chunk["country"] = chunk["country"].fillna("").astype(str).str.strip().str.upper()
    chunk["name_full_clean"] = chunk["business_name_original"].map(clean_name)
    chunk["name_core"] = chunk["name_full_clean"].map(make_name_core)
    chunk["name_latin"] = chunk["name_full_clean"].map(make_name_latin)
    chunk["address_clean"] = chunk["business_address_original"].map(clean_address)
    landmark_split = chunk["address_clean"].map(split_landmark)
    chunk["address_core"] = landmark_split.map(lambda pair: pair[0])
    chunk["landmark_text"] = landmark_split.map(lambda pair: pair[1])
    chunk["address_numbers"] = chunk["address_clean"].map(extract_numbers)
    chunk["postal_code"] = [extract_postal_code(address, country) for address, country in zip(chunk["address_clean"], chunk["country"])]
    chunk["name_tokens"] = chunk["name_full_clean"].map(tokenize)
    chunk["address_tokens"] = chunk["address_clean"].map(tokenize)
    STATS["total_rows"] += len(chunk)
    STATS["empty_name"] += int((chunk["name_full_clean"] == "").sum())
    STATS["empty_address"] += int((chunk["address_clean"] == "").sum())
    STATS["postal_code_found"] += int((chunk["postal_code"] != "").sum())
    STATS["landmark_found"] += int((chunk["landmark_text"] != "").sum())
    columns = ["entity_id", "business_name_original", "business_address_original", "name_full_clean", "name_core", "name_latin", "address_clean", "address_core", "landmark_text", "name_tokens", "address_tokens", "address_numbers", "postal_code", "country"]
    chunk = chunk[columns]
    temp_file = OUTPUT_DIR / f"part_{chunk_id:05d}.parquet.tmp"
    chunk.to_parquet(temp_file, engine="pyarrow", index=False, compression="zstd")
    temp_file.replace(final_file)
    done_file.write_text("DONE\n", encoding="utf-8")
def parse_args():
    parser = argparse.ArgumentParser(description="Advanced entity-resolution data cleaning")
    parser.add_argument("--input", type=Path, default=None)
    parser.add_argument("--output", type=Path, default=None)
    parser.add_argument("--chunk-size", type=int, default=None)
    parser.add_argument("--source-label", type=str, default="source1")
    return parser.parse_args()
def main():
    args = parse_args()
    global INPUT_FILE, OUTPUT_DIR, CHUNK_SIZE, PIPELINE_VERSION
    if args.input is not None:
        INPUT_FILE = args.input
    if args.output is not None:
        OUTPUT_DIR = args.output
    if args.chunk_size is not None:
        CHUNK_SIZE = args.chunk_size
    PIPELINE_VERSION = f"{args.source_label}_{BASE_PIPELINE_VERSION}"
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")
    validate_or_create_manifest()
    reader = pd.read_csv(INPUT_FILE, sep="\t", chunksize=CHUNK_SIZE, dtype=str, keep_default_na=False)
    chunk_id = 0
    for chunk in reader:
        process_chunk(chunk, chunk_id)
        chunk_id += 1
    OUTPUT_DIR.joinpath("_SUCCESS").write_text(f"Completed chunks: {chunk_id}\n", encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
def get_config():
    return {"pipeline_version": PIPELINE_VERSION, "input_file": str(INPUT_FILE), "input_size": INPUT_FILE.stat().st_size, "input_mtime": INPUT_FILE.stat().st_mtime, "chunk_size": CHUNK_SIZE, "required_columns": REQUIRED_COLUMNS}


def validate_or_create_manifest():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    manifest_path = OUTPUT_DIR / "manifest.json"
    current_config = get_config()
    if manifest_path.exists():
        with open(manifest_path, "r", encoding="utf-8") as f:
            old_config = json.load(f)
        for key in ["pipeline_version", "input_file", "input_size", "input_mtime", "chunk_size"]:
            if old_config.get(key) != current_config.get(key):
                raise RuntimeError(f"Manifest mismatch for '{key}'.")
    else:
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(current_config, f, indent=2)
    return manifest_path

In [ ]:
def process_chunk(chunk: pd.DataFrame, chunk_id: int):
    final_file = OUTPUT_DIR / f"part_{chunk_id:05d}.parquet"
    done_file = OUTPUT_DIR / f"part_{chunk_id:05d}.done"
    if final_file.exists() and done_file.exists():
        print(f"[SKIP] Chunk {chunk_id}")
        return
    missing = [column for column in REQUIRED_COLUMNS if column not in chunk.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")
    chunk["business_name_original"] = chunk["business_name"].fillna("").astype(str)
    chunk["business_address_original"] = chunk["business_address"].fillna("").astype(str)
    chunk["country"] = chunk["country"].fillna("").astype(str).str.strip().str.upper()
    chunk["name_full_clean"] = chunk["business_name_original"].map(clean_name)
    chunk["name_core"] = chunk["name_full_clean"].map(make_name_core)
    chunk["name_latin"] = chunk["name_full_clean"].map(make_name_latin)
    chunk["address_clean"] = chunk["business_address_original"].map(clean_address)
    landmark_split = chunk["address_clean"].map(split_landmark)
    chunk["address_core"] = landmark_split.map(lambda pair: pair[0])
    chunk["landmark_text"] = landmark_split.map(lambda pair: pair[1])
    chunk["address_numbers"] = chunk["address_clean"].map(extract_numbers)
    chunk["postal_code"] = [extract_postal_code(address, country) for address, country in zip(chunk["address_clean"], chunk["country"])]
    chunk["name_tokens"] = chunk["name_full_clean"].map(tokenize)
    chunk["address_tokens"] = chunk["address_clean"].map(tokenize)
    STATS["total_rows"] += len(chunk)
    STATS["empty_name"] += int((chunk["name_full_clean"] == "").sum())
    STATS["empty_address"] += int((chunk["address_clean"] == "").sum())
    STATS["postal_code_found"] += int((chunk["postal_code"] != "").sum())
    STATS["landmark_found"] += int((chunk["landmark_text"] != "").sum())
    columns = ["entity_id", "business_name_original", "business_address_original", "name_full_clean", "name_core", "name_latin", "address_clean", "address_core", "landmark_text", "name_tokens", "address_tokens", "address_numbers", "postal_code", "country"]
    chunk = chunk[columns]
    temp_file = OUTPUT_DIR / f"part_{chunk_id:05d}.parquet.tmp"
    chunk.to_parquet(temp_file, engine="pyarrow", index=False, compression="zstd")
    temp_file.replace(final_file)
    done_file.write_text("DONE\n", encoding="utf-8")

In [ ]:
def parse_args():
    parser = argparse.ArgumentParser(description="Advanced entity-resolution data cleaning")
    parser.add_argument("--input", type=Path, default=None)
    parser.add_argument("--output", type=Path, default=None)
    parser.add_argument("--chunk-size", type=int, default=None)
    parser.add_argument("--source-label", type=str, default="source1")
    return parser.parse_args()

In [ ]:
def main():
    args = parse_args()
    global INPUT_FILE, OUTPUT_DIR, CHUNK_SIZE, PIPELINE_VERSION
    if args.input is not None:
        INPUT_FILE = args.input
    if args.output is not None:
        OUTPUT_DIR = args.output
    if args.chunk_size is not None:
        CHUNK_SIZE = args.chunk_size
    PIPELINE_VERSION = f"{args.source_label}_{BASE_PIPELINE_VERSION}"
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")
    validate_or_create_manifest()
    reader = pd.read_csv(INPUT_FILE, sep="\t", chunksize=CHUNK_SIZE, dtype=str, keep_default_na=False)
    chunk_id = 0
    for chunk in reader:
        process_chunk(chunk, chunk_id)
        chunk_id += 1
    OUTPUT_DIR.joinpath("_SUCCESS").write_text(f"Completed chunks: {chunk_id}\n", encoding="utf-8")


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
import argparse
import json
import re
import unicodedata
import pandas as pd


# ============================================================
# OPTIONAL DEPENDENCY: transliteration
# ============================================================
# `unidecode` is MIT-licensed and purely local (no network calls, no
# business-identity lookups) - it's just a character transliteration
# table. Falls back to identity if it isn't installed.
try:
    from unidecode import unidecode as _transliterate
except ImportError:
    def _transliterate(text: str) -> str:
        return text

In [ ]:
# ============================================================
# CONFIG (defaults - overridable via CLI, see parse_args())
# ============================================================

INPUT_FILE = Path("dataset/train/train_source1.tsv")
OUTPUT_DIR = Path("processed/source1_cleaned")

CHUNK_SIZE = 100_000

REQUIRED_COLUMNS = [
    "entity_id",
    "business_name",
    "business_address",
    "country",
]

BASE_PIPELINE_VERSION = "cleaning_v2_advanced"
PIPELINE_VERSION = f"source1_{BASE_PIPELINE_VERSION}"  # reset per source in main()
